# Day 10 — Selecting columns by name, regex and dtype

*100 Days of Polars · Phase 2: Expressions*

- **Post:** {{TODO: Medium link}}
- **Dataset:** [Olist Brazilian e-commerce](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce) — CSVs in `data/raw/`
- **Written for:** Polars 2.x (run from the repo root)

In [1]:
import polars as pl

print("polars", pl.__version__)

polars 2.0.0


`pl.col` does more than pick one column. It takes names, lists, **regular expressions** and **dtypes**, and expands to as many columns as match.

In [2]:
orders = pl.read_csv("data/raw/olist_orders_dataset.csv", try_parse_dates=True)
orders.head(3)

order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
str,str,str,datetime[μs],datetime[μs],datetime[μs],datetime[μs],datetime[μs]
"""e481f51cbdc54678b7cc49136f2d6a…","""9ef432eb6251297304e76186b10a92…","""delivered""",2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
"""53cdb2fc8bc7dce0b6741e21502734…","""b0830fb4747a6c6d20dea0b8c802d7…","""delivered""",2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
"""47770eb9100c2d0c44946d9cf07ec6…","""41ce2a54c0b03bf3443c3d931a3670…","""delivered""",2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00


In [3]:
products = pl.read_csv("data/raw/olist_products_dataset.csv")
products.head(3)

product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
str,str,i64,i64,i64,i64,i64,i64,i64
"""1e9e8ef04dbcff4541ed26657ea517…","""perfumaria""",40,287,1,225,16,10,14
"""3aa071139cb16b67ca9e5dea641aaa…","""artes""",44,276,1,1000,30,18,20
"""96bd76ec8810374ed1b65e29197571…","""esporte_lazer""",46,250,1,154,18,9,15


## 1. By name

In [4]:
orders.select(pl.col("order_id"), pl.col("order_status")).head(3)

order_id,order_status
str,str
"""e481f51cbdc54678b7cc49136f2d6a…","""delivered"""
"""53cdb2fc8bc7dce0b6741e21502734…","""delivered"""
"""47770eb9100c2d0c44946d9cf07ec6…","""delivered"""


In [5]:
# Several names in one pl.col (or a list) — same result
orders.select(pl.col("order_id", "order_status")).head(3)

order_id,order_status
str,str
"""e481f51cbdc54678b7cc49136f2d6a…","""delivered"""
"""53cdb2fc8bc7dce0b6741e21502734…","""delivered"""
"""47770eb9100c2d0c44946d9cf07ec6…","""delivered"""


## 2. By regex

A string is treated as a regex **only** when it starts with `^` and ends with `$`.

In [6]:
orders.select(pl.col("^order_.*_date$")).head(3)

order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
datetime[μs],datetime[μs],datetime[μs]
2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00


In [7]:
# The Olist products table has a typo baked into two column names: 'lenght'
products.select(pl.col("^.*_lenght$")).head(3)

product_name_lenght,product_description_lenght
i64,i64
40,287
44,276
46,250


In [8]:
# Gotcha: without ^...$ it is a literal column name
try:
    orders.select(pl.col("order_.*_date"))
except Exception as err:
    print(type(err).__name__, "->", str(err).splitlines()[0])

ColumnNotFoundError -> unable to find column "order_.*_date"; valid columns: ["order_id", "customer_id", "order_status", "order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date", "order_delivered_customer_date", "order_estimated_delivery_date"]


## 3. By dtype

In [9]:
print(orders.schema)
orders.select(pl.col(pl.Datetime)).head(3)  # every datetime column, any time unit

Schema({'order_id': String, 'customer_id': String, 'order_status': String, 'order_purchase_timestamp': Datetime(time_unit='us', time_zone=None), 'order_approved_at': Datetime(time_unit='us', time_zone=None), 'order_delivered_carrier_date': Datetime(time_unit='us', time_zone=None), 'order_delivered_customer_date': Datetime(time_unit='us', time_zone=None), 'order_estimated_delivery_date': Datetime(time_unit='us', time_zone=None)})


order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
datetime[μs],datetime[μs],datetime[μs],datetime[μs],datetime[μs]
2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00


In [10]:
items = pl.read_csv("data/raw/olist_order_items_dataset.csv", try_parse_dates=True)
print(items.schema)
items.select(pl.col(pl.Float64)).head(3)  # price and freight_value

Schema({'order_id': String, 'order_item_id': Int64, 'product_id': String, 'seller_id': String, 'shipping_limit_date': Datetime(time_unit='us', time_zone=None), 'price': Float64, 'freight_value': Float64})


price,freight_value
f64,f64
58.9,13.29
239.9,19.93
199.0,17.87


In [11]:
# products has no Float64 columns at all: what does a dtype selection with no match return?
print(products.schema)
products.select(pl.col(pl.Float64)).shape

Schema({'product_id': String, 'product_category_name': String, 'product_name_lenght': Int64, 'product_description_lenght': Int64, 'product_photos_qty': Int64, 'product_weight_g': Int64, 'product_length_cm': Int64, 'product_height_cm': Int64, 'product_width_cm': Int64})


(0, 0)

In [12]:
# Several dtypes at once
products.select(pl.col(pl.String, pl.Int64)).head(3)

product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
str,str,i64,i64,i64,i64,i64,i64,i64
"""1e9e8ef04dbcff4541ed26657ea517…","""perfumaria""",40,287,1,225,16,10,14
"""3aa071139cb16b67ca9e5dea641aaa…","""artes""",44,276,1,1000,30,18,20
"""96bd76ec8810374ed1b65e29197571…","""esporte_lazer""",46,250,1,154,18,9,15


## 4. One operation, many columns

Because `pl.col` expands, whatever you chain onto it runs on every matched column.

In [13]:
orders.select(
    "order_id",
    pl.col("^order_.*_date$").dt.date(),  # three datetime columns -> dates, in one line
).head(3)

order_id,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
str,date,date,date
"""e481f51cbdc54678b7cc49136f2d6a…",2017-10-04,2017-10-10,2017-10-18
"""53cdb2fc8bc7dce0b6741e21502734…",2018-07-26,2018-08-07,2018-08-13
"""47770eb9100c2d0c44946d9cf07ec6…",2018-08-08,2018-08-17,2018-09-04


In [14]:
# Mean dimensions of products: three columns, one expression
products.select(pl.col("^product_.*_cm$").mean().round(1))

product_length_cm,product_height_cm,product_width_cm
f64,f64,f64
30.8,16.9,23.2


## 5. pandas comparison

| pandas | Polars |
|---|---|
| `df[["a", "b"]]` | `df.select(pl.col("a", "b"))` or `df.select("a", "b")` |
| `df.filter(regex="_date$")` | `df.select(pl.col("^.*_date$"))` |
| `df.select_dtypes("float")` | `df.select(pl.col(pl.Float64))` |

In Polars all three are the same `pl.col`, and the result is an expression you can keep transforming.

## Try it yourself

1. Select every column of `orders` that ends in `_at` or `_date` with one regex.
2. Round every `Float64` column of `items` to 0 decimals in a `with_columns`.
3. What happens with `pl.col(pl.Datetime("ms"))` on `orders`? Check `orders.schema` for the time unit.

## Appendix: Regex 101 (just enough for column names)

A regex is a pattern that describes text. For picking columns you need only a handful of pieces.

**Positions and characters**

| Piece | Means | Example | Matches |
|---|---|---|---|
| `^` | start of the name | `^order_` | names starting with `order_` |
| `$` | end of the name | `_date$` | names ending with `_date` |
| `.` | any one character | `^c.t$` | `cat`, `cut` |
| `\d` | one digit | `^q\d$` | `q1`, `q4` |
| `[0-9]` | one digit, strictly 0–9 | `^q[0-9]$` | `q1`, `q4` |
| `\w` | one "word" character: letter, digit or `_` | `^\w+$` | any plain column name |
| `\s` | one whitespace character | `^order\sid$` | `order id` |
| `\D` `\W` `\S` | uppercase = the opposite | `\D` | any non-digit |
| `\.` | a literal dot | `v1\.0` | `v1.0` only |

**Repeats: they apply to the ONE piece right before them**

| Symbol | How many times | Example | Matches |
|---|---|---|---|
| `*` | 0 or more | `ab*` | `a`, `ab`, `abb` … |
| `+` | 1 or more | `ab+` | `ab`, `abb` … (not `a`) |
| `?` | 0 or 1 (optional) | `ab?` | `a`, `ab` |
| `{4}` | exactly 4 | `\d{4}` | `2017` |

"The piece before" is one character, a class like `\d`, or a group in parentheses: `ab*` repeats only `b`, `(ab)*` repeats `ab`. That is why "anything" is `.*`: any character (`.`), any number of times (`*`).

**Choices**

| Piece | Means | Example | Matches |
|---|---|---|---|
| `\|` | or | `cm\|g` | `cm` or `g` |
| `( )` | group, to limit the or (or a repeat) to one part | `_(cm\|g)$` | ends with `_cm` or `_g` |

Two Polars rules: the pattern must start with `^` and end with `$` (otherwise it is a literal column name), and it must match the **whole** name.

`^order_.*_date$` reads: start, `order_`, anything, `_date`, end.

In [15]:
def show(df, pattern):
    """Print which columns of df a pattern selects."""
    try:
        print(f"{pattern:<38} -> {df.select(pl.col(pattern)).columns}")
    except Exception as err:
        print(f"{pattern:<38} -> {type(err).__name__}")

print(orders.columns)

['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']


In [16]:
show(orders, "^order_.*_date$")      # starts with order_, ends with _date
show(orders, "^order_.*$")           # starts with order_
show(orders, "^.*_date$")            # ends with _date
show(orders, "^.*_id$")              # the id columns
show(orders, "^order_delivered_.*$") # the two delivery timestamps

^order_.*_date$                        -> ['order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']
^order_.*$                             -> ['order_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']
^.*_date$                              -> ['order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']
^.*_id$                                -> ['order_id', 'customer_id']
^order_delivered_.*$                   -> ['order_delivered_carrier_date', 'order_delivered_customer_date']


**The most common mistake: `*` is not a wildcard on its own.** In file searches (`ls order_*_date.csv`), `*` means "anything". In regex, `*` only repeats the piece before it. So `^order_*_date$` means `order` + zero or more `_` + `_date`: it matches `order_date` or `order__date`, and none of our columns.

| Name | `^order_.*_date$` | `^order_*_date$` |
|---|---|---|
| `order_delivered_carrier_date` | ✅ | ❌ |
| `order_date` | ✅ (`.*` matches nothing) | ✅ (`_*` matches nothing) |
| `order__date` | ✅ | ✅ |

In [17]:
show(orders, "^order_.*_date$")   # glob "*" becomes ".*" in regex
show(orders, "^order_*_date$")    # * repeats only the underscore -> no match

^order_.*_date$                        -> ['order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']
^order_*_date$                         -> []


In [18]:
# | means "or"; ( ) keeps the or to one part of the pattern
show(orders, "^order_(approved_at|purchase_timestamp)$")
show(orders, "^.*(_at|_date|_timestamp)$")   # every time column, whatever its suffix
show(products, "^product_.*_(cm|g)$")        # dimensions and weight
show(products, "^product_(name|description)_lenght$")

^order_(approved_at|purchase_timestamp)$ -> ['order_purchase_timestamp', 'order_approved_at']
^.*(_at|_date|_timestamp)$             -> ['order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']
^product_.*_(cm|g)$                    -> ['product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']
^product_(name|description)_lenght$    -> ['product_name_lenght', 'product_description_lenght']


In [19]:
# .* allows nothing in between, .+ needs at least one character
show(orders, "^order_.*id$")   # order_id: .* matches the empty text before 'id'
show(orders, "^order_.+id$")   # needs at least one character after order_ -> no match

^order_.*id$                           -> ['order_id']
^order_.+id$                           -> []


In [20]:
# \d = a digit. Handy for wide tables with years as column names.
# Use a raw string r"..." so Python leaves the backslash alone.
sales = pl.DataFrame({"sales_2017": [1], "sales_2018": [2], "sales_total": [3]})
show(sales, r"^sales_\d+$")    # the year columns only
show(sales, r"^sales_\d{4}$")  # exactly four digits

^sales_\d+$                            -> ['sales_2017', 'sales_2018']
^sales_\d{4}$                          -> ['sales_2017', 'sales_2018']


In [21]:
# * + ? on one piece, tested on plain names with Python's re (fullmatch = implicit ^...$)
import re
names = ["a", "ab", "abb", "abab", "q1", "q12", "q2017"]
for p in ["ab*", "ab+", "ab?", "(ab)*", r"q\d", r"q\d+", r"q\d{4}"]:
    print(f"{p:<8} -> {[n for n in names if re.fullmatch(p, n)]}")

ab*      -> ['a', 'ab', 'abb']
ab+      -> ['ab', 'abb']
ab?      -> ['a', 'ab']
(ab)*    -> ['ab', 'abab']
q\d      -> ['q1']
q\d+     -> ['q1', 'q12', 'q2017']
q\d{4}   -> ['q2017']


`\d` follows Unicode in both Polars and Python, so it also matches digits from other scripts (for example Arabic-Indic `٣`). Irrelevant for column names; use `[0-9]` when you need strictly 0–9.

Polars uses the Rust `regex` engine. For simple patterns like these it behaves like Python's `re`, so you can try a pattern on a plain list of names first:

In [22]:
import re

[c for c in orders.columns if re.fullmatch(r"order_.*_date", c)]  # fullmatch = implicit ^...$

['order_delivered_carrier_date',
 'order_delivered_customer_date',
 'order_estimated_delivery_date']

**Practice:** write one pattern for each, check it with `show`:

1. Every `orders` column with `delivered` anywhere in the name.
2. Every `products` column that ends in `_lenght` or `_qty`.
3. Every `orders` column that does **not** start with `order_`. (Hint: no simple regex for "not"; try `pl.exclude("^order_.*$")` or a selector instead.)